In [0]:
%pip install -U databricks-ai-search databricks-sdk

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.2 MB/s  0:00:00
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.163.0
    Uninstalling databricks-sdk-0.163.0:
      Successfully uninstalled databricks-sdk-0.163.0
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
CATALOG = "learn_get_started_genai_tshivakrishnareddy123"  # Change to a catalog you can use
SCHEMA = "genai_support_demo"

SEARCH_ENDPOINT = "vs_endpoint_1"

EMBEDDING_ENDPOINT = "databricks-gte-large-en"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

TABLE_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks_index"

assert not EMBEDDING_ENDPOINT.startswith("YOUR_"), \
    "Set EMBEDDING_ENDPOINT first."
assert not LLM_ENDPOINT.startswith("YOUR_"), \
    "Set LLM_ENDPOINT first."

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print("Source table:", TABLE_NAME)
print("Search index:", INDEX_NAME)

Source table: learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks
Search index: learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index


In [0]:
documents = [
    {
        "doc_id": "INC-001",
        "title": "Talend job: Hive connection timeout",
        "source": "demo/INC-001",
        "text": """
        Symptom: A Talend job could not connect to HiveServer2 and timed out.
        Investigation: Check HiveServer2 health, network reachability from
        the Talend execution host, JDBC settings, and metastore database
        availability. Review HiveServer2 and database logs around the failure.
        Historical cause: The metastore database was unavailable.
        Resolution: The platform team followed its approved database recovery
        procedure, verified Hive connectivity, and then reran the failed job.
        A connection timeout alone does not establish the root cause.
        """
    },
    {
        "doc_id": "INC-002",
        "title": "Spark job: executor out-of-memory",
        "source": "demo/INC-002",
        "text": """
        Symptom: Spark executors failed with out-of-memory errors during a join.
        Investigation: Inspect failed stages, task sizes, partition skew,
        executor logs, and the physical query plan.
        Historical cause: A skewed join created very large partitions.
        Resolution: The team reduced unnecessary columns, addressed skew,
        and reviewed partition sizing before retrying the workload.
        """
    },
    {
        "doc_id": "INC-003",
        "title": "Talend job: database authentication failure",
        "source": "demo/INC-003",
        "text": """
        Symptom: A Talend database connection returned authentication failed.
        Investigation: Check whether the service account is locked or expired,
        confirm the job uses the intended context, and check whether a recent
        credential rotation was reflected in the configured secret.
        Historical cause: The job referenced an outdated credential.
        Resolution: The team updated the authorized secret reference and
        validated connectivity. Never print passwords in job logs.
        """
    }
]

print(f"Loaded {len(documents)} sample documents.")

Loaded 3 sample documents.


In [0]:
print(f"Loaded {len(documents)} sample documents.")

Loaded 3 sample documents.


In [0]:
import json

for doc in documents:
    print("=" * 80)
    print(f"doc_id : {doc['doc_id']}")
    print(f"title  : {doc['title']}")
    print(f"source : {doc['source']}")
    print(f"text   :\n{doc['text'].strip()}")
print("=" * 80)
print(f"Total: {len(documents)} documents")

doc_id : INC-001
title  : Talend job: Hive connection timeout
source : demo/INC-001
text   :
Symptom: A Talend job could not connect to HiveServer2 and timed out.
        Investigation: Check HiveServer2 health, network reachability from
        the Talend execution host, JDBC settings, and metastore database
        availability. Review HiveServer2 and database logs around the failure.
        Historical cause: The metastore database was unavailable.
        Resolution: The platform team followed its approved database recovery
        procedure, verified Hive connectivity, and then reran the failed job.
        A connection timeout alone does not establish the root cause.
doc_id : INC-002
title  : Spark job: executor out-of-memory
source : demo/INC-002
text   :
Symptom: Spark executors failed with out-of-memory errors during a join.
        Investigation: Inspect failed stages, task sizes, partition skew,
        executor logs, and the physical query plan.
        Historical cause: A 

In [0]:
import hashlib

def split_text(text, chunk_words=100, overlap_words=20):
    if not 0 <= overlap_words < chunk_words:
        raise ValueError("Overlap must be smaller than chunk size.")

    words = text.split()
    step = chunk_words - overlap_words

    for start in range(0, len(words), step):
        yield " ".join(words[start:start + chunk_words])

        if start + chunk_words >= len(words):
            break

rows = []

for doc in documents:
    for position, chunk in enumerate(split_text(doc["text"])):
        chunk_id = hashlib.sha256(
            f"{doc['doc_id']}:{position}".encode()
        ).hexdigest()

        rows.append((
            chunk_id,
            doc["doc_id"],
            doc["title"],
            doc["source"],
            f"{doc['title']}\n{chunk}"
        ))

chunks_df = spark.createDataFrame(
    rows,
    """
    chunk_id STRING, doc_id STRING, title STRING,
    source STRING, chunk_text STRING
    """
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    chunk_id STRING,
    doc_id STRING,
    title STRING,
    source STRING,
    chunk_text STRING
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

chunks_df.createOrReplaceTempView("incoming_demo_chunks")

spark.sql(f"""
MERGE INTO {TABLE_NAME} AS target
USING incoming_demo_chunks AS incoming
ON target.chunk_id = incoming.chunk_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.table(TABLE_NAME))

chunk_id,doc_id,title,source,chunk_text
c5e3c8f276a7bac4a386b47780df21a996d43b561f9c051a344a8e414ce69807,INC-003,Talend job: database authentication failure,demo/INC-003,"Talend job: database authentication failure Symptom: A Talend database connection returned authentication failed. Investigation: Check whether the service account is locked or expired, confirm the job uses the intended context, and check whether a recent credential rotation was reflected in the configured secret. Historical cause: The job referenced an outdated credential. Resolution: The team updated the authorized secret reference and validated connectivity. Never print passwords in job logs."
730c019d76735cb43df9843d2bd3a234603a24c4c55da1bcf8a249bca227031c,INC-002,Spark job: executor out-of-memory,demo/INC-002,"Spark job: executor out-of-memory Symptom: Spark executors failed with out-of-memory errors during a join. Investigation: Inspect failed stages, task sizes, partition skew, executor logs, and the physical query plan. Historical cause: A skewed join created very large partitions. Resolution: The team reduced unnecessary columns, addressed skew, and reviewed partition sizing before retrying the workload."
80425844c248b428ca30ba392274735083f74b0ca56f544adaeacf33ed96fc7d,INC-001,Talend job: Hive connection timeout,demo/INC-001,"Talend job: Hive connection timeout Symptom: A Talend job could not connect to HiveServer2 and timed out. Investigation: Check HiveServer2 health, network reachability from the Talend execution host, JDBC settings, and metastore database availability. Review HiveServer2 and database logs around the failure. Historical cause: The metastore database was unavailable. Resolution: The platform team followed its approved database recovery procedure, verified Hive connectivity, and then reran the failed job. A connection timeout alone does not establish the root cause."


In [0]:
SEARCH_ENDPOINT = "support-rag-demo-endpoint"

In [0]:
from datetime import timedelta
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient()

if not search_client.endpoint_exists(SEARCH_ENDPOINT):
    search_client.create_endpoint(
        name=SEARCH_ENDPOINT,
        endpoint_type="STANDARD"
    )

search_client.wait_for_endpoint(
    name=SEARCH_ENDPOINT,
    verbose=True,
    timeout=timedelta(minutes=30)
)

print("Search endpoint is ready.")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Endpoint support-rag-demo-endpoint is ONLINE.
Search endpoint is ready.


In [0]:
if not search_client.index_exists(index_name=INDEX_NAME):
    index = search_client.create_delta_sync_index(
        endpoint_name=SEARCH_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=TABLE_NAME,
        primary_key="chunk_id",
        pipeline_type="TRIGGERED",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT
    )
else:
    index = search_client.get_index(
        endpoint_name=SEARCH_ENDPOINT,
        index_name=INDEX_NAME
    )
    index.wait_until_ready(timeout=timedelta(minutes=30))
    index.sync()

index.wait_until_ready(
    verbose=True,
    timeout=timedelta(minutes=30),
    wait_for_updates=True
)

print("Index is ready.")

Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_UPDATING_PIPELINE_RESOURCES. Time: 0s.
Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_TRIGGERED_UPDATE. Time: 31s.
Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_TRIGGERED_UPDATE. Time: 61s.
Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_TRIGGERED_UPDATE. Time: 93s.
Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_NO_PENDING_UPDATE.
Index is ready.


In [0]:
def retrieve_incidents(question, top_k=3):
    result = index.similarity_search(
        query_text=question,
        columns=["doc_id", "title", "source", "chunk_text"],
        query_type="HYBRID",
        num_results=top_k
    )

    column_names = [
        column["name"]
        for column in result["manifest"]["columns"]
    ]

    return [
        dict(zip(column_names, row))
        for row in result.get("result", {}).get("data_array", [])
    ]

question = "My Talend job failed with a Hive connection timeout. What should I check?"

hits = retrieve_incidents(question)

for hit in hits:
    print(f"\nDocument: {hit['doc_id']} — {hit['title']}")
    print(hit["chunk_text"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

Document: INC-001 — Talend job: Hive connection timeout
Talend job: Hive connection timeout
Symptom: A Talend job could not connect to HiveServer2 and timed out. Investigation: Check HiveServer2 health, network reachability from the Talend execution host, JDBC settings, and metastore database availability. Review HiveServer2 and database logs around the failure. Historical cause: The metastore database was unavailable. Resolution: The platform team followed its approved database recovery procedure, verified Hive connectivity, and then reran the failed job. A connection timeout alone does not establish the root cause.

Document: INC-003 — Talend job: database authentication failure
Talend job: database authentication failure
Symptom: A Talend database connection returned authenti

In [0]:
from datetime import timedelta
from databricks.ai_search.client import AISearchClient

# Use the same names you used when creating the index
SEARCH_ENDPOINT = "support-rag-demo-endpoint"
INDEX_NAME = "learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks"

search_client = AISearchClient()

# Connect to an existing index
index = search_client.get_index(
    endpoint_name=SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

index.wait_until_ready(
    verbose=True,
    timeout=timedelta(minutes=30)
)

print("Index connected and ready.")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


---------------------------------------------------------------------------
BadRequest                                Traceback (most recent call last)
File <command-5736053456701274>, line 11
      8 search_client = AISearchClient()
     10 # Connect to an existing index
---> 11 index = search_client.get_index(
     12     endpoint_name=SEARCH_ENDPOINT,
     13     index_name=INDEX_NAME
     14 )
     16 index.wait_until_ready(
     17     verbose=True,
     18     timeout=timedelta(minutes=30)
     19 )
     21 print("Index connected and ready.")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-063ea3f2-7229-423b-aa62-48599e35fa21/lib/python3.12/site-packages/databricks/ai_search/client.py:1038, in AISearchClient.get_index(self, endpoint_name, index_name)
   1029 def get_index(self, endpoint_name=None, index_name=None):
   1030     """
   1031     Get an index.
   1032 
   (...)   1036     :raises PermissionDenied: If user lacks permission to view the index.
   1037     """
-> 1038  

In [0]:
CATALOG = "learn_get_started_genai_tshivakrishnareddy123"
SCHEMA = "genai_support_demo"

TABLE_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks_index"

SEARCH_ENDPOINT = "support-rag-demo-endpoint"

# Replace with an embedding endpoint available in your workspace
EMBEDDING_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()

for endpoint in workspace.serving_endpoints.list():
    print(endpoint.name)

databricks-gpt-oss-120b
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-qwen35-122b-a10b
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b
databricks-llama-4-maverick


In [0]:
from datetime import timedelta
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient()

if search_client.index_exists(index_name=INDEX_NAME):
    index = search_client.get_index(
        endpoint_name=SEARCH_ENDPOINT,
        index_name=INDEX_NAME
    )
else:
    if EMBEDDING_ENDPOINT == "YOUR_EMBEDDING_ENDPOINT":
        raise ValueError(
            "Set EMBEDDING_ENDPOINT to your actual embedding endpoint name."
        )

    # Enable change tracking on the source table
    spark.sql(f"""
        ALTER TABLE {TABLE_NAME}
        SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)

    index = search_client.create_delta_sync_index(
        endpoint_name=SEARCH_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=TABLE_NAME,
        primary_key="chunk_id",
        pipeline_type="TRIGGERED",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT
    )

index.wait_until_ready(
    verbose=True,
    timeout=timedelta(minutes=30),
    wait_for_updates=True
)

print(f"Search index ready: {INDEX_NAME}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_NO_PENDING_UPDATE.
Search index ready: learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index


In [0]:
hits = retrieve_incidents(
    "My Talend job failed with a Hive connection timeout. "
    "What should I check?"
)

for hit in hits:
    print(f"\n{hit['doc_id']} — {hit['title']}")
    print(hit["chunk_text"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

INC-001 — Talend job: Hive connection timeout
Talend job: Hive connection timeout
Symptom: A Talend job could not connect to HiveServer2 and timed out. Investigation: Check HiveServer2 health, network reachability from the Talend execution host, JDBC settings, and metastore database availability. Review HiveServer2 and database logs around the failure. Historical cause: The metastore database was unavailable. Resolution: The platform team followed its approved database recovery procedure, verified Hive connectivity, and then reran the failed job. A connection timeout alone does not establish the root cause.

INC-003 — Talend job: database authentication failure
Talend job: database authentication failure
Symptom: A Talend database connection returned authentication failed. Inves

In [0]:
def retrieve_incidents(question, top_k=3):
    result = index.similarity_search(
        query_text=question,
        columns=["doc_id", "title", "source", "chunk_text"],
        query_type="HYBRID",
        num_results=top_k
    )

    column_names = [
        column["name"]
        for column in result["manifest"]["columns"]
    ]

    return [
        dict(zip(column_names, row))
        for row in result.get("result", {}).get("data_array", [])
    ]

In [0]:
question = (
    "My Talend job failed with a Hive connection timeout. "
    "What should I check?"
)

hits = retrieve_incidents(question)

for hit in hits:
    print("=" * 80)
    print(f"Document: {hit['doc_id']} — {hit['title']}")
    print(hit["chunk_text"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Document: INC-001 — Talend job: Hive connection timeout
Talend job: Hive connection timeout
Symptom: A Talend job could not connect to HiveServer2 and timed out. Investigation: Check HiveServer2 health, network reachability from the Talend execution host, JDBC settings, and metastore database availability. Review HiveServer2 and database logs around the failure. Historical cause: The metastore database was unavailable. Resolution: The platform team followed its approved database recovery procedure, verified Hive connectivity, and then reran the failed job. A connection timeout alone does not establish the root cause.
Document: INC-003 — Talend job: database authentication failure
Talend job: database authentication failure
Symptom: A Talend database connection returned authentica

In [0]:
from datetime import timedelta
from databricks.ai_search.client import AISearchClient

CATALOG = "learn_get_started_genai_tshivakrishnareddy123"
SCHEMA = "genai_support_demo"

INDEX_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks_index"
SEARCH_ENDPOINT = "support-rag-demo-endpoint"

# Connect to the existing AI Search index
search_client = AISearchClient()

index = search_client.get_index(
    endpoint_name=SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

index.wait_until_ready(
    verbose=True,
    timeout=timedelta(minutes=30)
)

# Define the retrieval function
def retrieve_incidents(question, top_k=3):
    result = index.similarity_search(
        query_text=question,
        columns=["doc_id", "title", "source", "chunk_text"],
        query_type="HYBRID",
        num_results=top_k
    )

    column_names = [
        column["name"]
        for column in result["manifest"]["columns"]
    ]

    return [
        dict(zip(column_names, row))
        for row in result.get("result", {}).get("data_array", [])
    ]

print(f"retrieve_incidents is ready to query: {INDEX_NAME}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Index learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index is in state ONLINE_NO_PENDING_UPDATE.
retrieve_incidents is ready to query: learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.incident_chunks_index


In [0]:
CATALOG = "learn_get_started_genai_tshivakrishnareddy123"
SCHEMA = "genai_support_demo"

SEARCH_ENDPOINT = "support-rag-demo-endpoint"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks_index"

create_function_sql = f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.retrieve_incidents(
    question STRING
)
RETURNS ARRAY<STRUCT<
    doc_id STRING,
    title STRING,
    source STRING,
    chunk_text STRING
>>
LANGUAGE PYTHON
ENVIRONMENT (
    dependencies = '["databricks-ai-search==0.83", "databricks-sdk"]',
    environment_version = '6'
)
AS $$
    from databricks.ai_search.client import AISearchClient

    search_client = AISearchClient()

    index = search_client.get_index(
        endpoint_name="{SEARCH_ENDPOINT}",
        index_name="{INDEX_NAME}"
    )

    result = index.similarity_search(
        query_text=question,
        columns=["doc_id", "title", "source", "chunk_text"],
        query_type="HYBRID",
        num_results=3
    )

    column_names = [
        column["name"]
        for column in result["manifest"]["columns"]
    ]

    return [
        dict(zip(column_names, row))
        for row in result.get("result", {{}}).get("data_array", [])
    ]
$$
"""

spark.sql(create_function_sql)

print(f"Created function: {CATALOG}.{SCHEMA}.retrieve_incidents")

# Test the UC function
result_df = spark.sql(f"""
    SELECT {CATALOG}.{SCHEMA}.retrieve_incidents(
        "My Talend job failed with a Hive connection timeout. What should I check?"
    ) AS hits
""")

display(result_df)


Created function: learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.retrieve_incidents


{"ts": "2026-11-11 11:29:36.095", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[UDF_USER_CODE_ERROR.GENERIC] Execution of function learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.retrieve_incidents failed. \n== Error ==\nMlflowException: Reading Databricks credential configuration failed with MLflow tracking URI 'None'. Please ensure that the 'databricks-sdk' PyPI library is installed, the tracking URI is set correctly, and Databricks authentication is properly configured. The tracking URI can be either 'databricks' (using profile name specified by 'DATABRICKS_CONFIG_PROFILE' environment variable or using 'DEFAULT' authentication profile if 'DATABRICKS_CONFIG_PROFILE' environment variable does not exist) or 'databricks://{profile}'. You can configure Da

---------------------------------------------------------------------------
SparkRuntimeException                     Traceback (most recent call last)
File <command-5736053456701282>, line 62
     55 # Test the UC function
     56 result_df = spark.sql(f"""
     57     SELECT {CATALOG}.{SCHEMA}.retrieve_incidents(
     58         "My Talend job failed with a Hive connection timeout. What should I check?"
     59     ) AS hits
     60 """)
---> 62 display(result_df)

File /databricks/python_shell/lib/dbruntime/display.py:209, in Display.display(self, input, *args, **kwargs)
    207     pass
    208 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 209     self.display_connect_table(input, **kwargs)
    210 elif isinstance(input, ConnectDataFrame):
    211     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:115, in Display.display_connect_table(self, df, **kwargs)
    112     self.cf_helper.display_streaming_dataframe(df, config,

In [0]:
CATALOG = "learn_get_started_genai_tshivakrishnareddy123"
SCHEMA = "genai_support_demo"

INDEX_NAME = f"{CATALOG}.{SCHEMA}.incident_chunks_index"

create_function_sql = f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.retrieve_incidents(
    question STRING
)
RETURNS ARRAY<STRUCT<
    doc_id: STRING,
    title: STRING,
    source: STRING,
    chunk_text: STRING
>>
LANGUAGE SQL
RETURN
    SELECT collect_list(
        named_struct(
            'doc_id', CAST(matches.doc_id AS STRING),
            'title', CAST(matches.title AS STRING),
            'source', CAST(matches.source AS STRING),
            'chunk_text', CAST(matches.chunk_text AS STRING)
        )
    )
    FROM vector_search(
        index => '{INDEX_NAME}',
        query_text => question,
        query_type => 'HYBRID',
        num_results => 3
    ) AS matches
"""

spark.sql(create_function_sql)

print(f"Created: {CATALOG}.{SCHEMA}.retrieve_incidents")

Created: learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.retrieve_incidents


In [0]:
spark.sql("""
DROP FUNCTION IF EXISTS
learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.retrieve_incidents
""")

DataFrame[]

In [0]:
result_df = spark.sql(f"""
    SELECT {CATALOG}.{SCHEMA}.retrieve_incidents(
        'My Talend job failed with a Hive connection timeout. What should I check?'
    ) AS hits
""")

display(result_df)

hits
"List(List(INC-001, Talend job: Hive connection timeout, demo/INC-001, Talend job: Hive connection timeout Symptom: A Talend job could not connect to HiveServer2 and timed out. Investigation: Check HiveServer2 health, network reachability from the Talend execution host, JDBC settings, and metastore database availability. Review HiveServer2 and database logs around the failure. Historical cause: The metastore database was unavailable. Resolution: The platform team followed its approved database recovery procedure, verified Hive connectivity, and then reran the failed job. A connection timeout alone does not establish the root cause.), List(INC-003, Talend job: database authentication failure, demo/INC-003, Talend job: database authentication failure Symptom: A Talend database connection returned authentication failed. Investigation: Check whether the service account is locked or expired, confirm the job uses the intended context, and check whether a recent credential rotation was reflected in the configured secret. Historical cause: The job referenced an outdated credential. Resolution: The team updated the authorized secret reference and validated connectivity. Never print passwords in job logs.), List(INC-002, Spark job: executor out-of-memory, demo/INC-002, Spark job: executor out-of-memory Symptom: Spark executors failed with out-of-memory errors during a join. Investigation: Inspect failed stages, task sizes, partition skew, executor logs, and the physical query plan. Historical cause: A skewed join created very large partitions. Resolution: The team reduced unnecessary columns, addressed skew, and reviewed partition sizing before retrying the workload.))"


In [0]:
%sql
SELECT learn_get_started_genai_tshivakrishnareddy123.genai_support_demo.retrieve_incidents(
  'My Talend job failed with a Hive connection timeout. What should I check?'
) AS hits;

hits
"List(List(INC-001, Talend job: Hive connection timeout, demo/INC-001, Talend job: Hive connection timeout Symptom: A Talend job could not connect to HiveServer2 and timed out. Investigation: Check HiveServer2 health, network reachability from the Talend execution host, JDBC settings, and metastore database availability. Review HiveServer2 and database logs around the failure. Historical cause: The metastore database was unavailable. Resolution: The platform team followed its approved database recovery procedure, verified Hive connectivity, and then reran the failed job. A connection timeout alone does not establish the root cause.), List(INC-003, Talend job: database authentication failure, demo/INC-003, Talend job: database authentication failure Symptom: A Talend database connection returned authentication failed. Investigation: Check whether the service account is locked or expired, confirm the job uses the intended context, and check whether a recent credential rotation was reflected in the configured secret. Historical cause: The job referenced an outdated credential. Resolution: The team updated the authorized secret reference and validated connectivity. Never print passwords in job logs.), List(INC-002, Spark job: executor out-of-memory, demo/INC-002, Spark job: executor out-of-memory Symptom: Spark executors failed with out-of-memory errors during a join. Investigation: Inspect failed stages, task sizes, partition skew, executor logs, and the physical query plan. Historical cause: A skewed join created very large partitions. Resolution: The team reduced unnecessary columns, addressed skew, and reviewed partition sizing before retrying the workload.))"
